In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

Load dataset

In [ ]:
df = pd.read_csv('water_potability.csv')
print("(row, column)")
print(df.shape)
print("\n(Missing Values)")
print(df.isnull().sum())

(row, column)
(3276, 10)

(Missing Values)
ph                 491
Hardness             0
Solids               0
Chloramines          0
Sulfate            781
Conductivity         0
Organic_carbon       0
Trihalomethanes    162
Turbidity            0
Potability           0
dtype: int64


Data Imputation

In [ ]:
df['ph'] = df['ph'].fillna(df['ph'].median())
df['Sulfate'] = df['Sulfate'].fillna(df['Sulfate'].median())
df['Trihalomethanes'] = df['Trihalomethanes'].fillna(df['Trihalomethanes'].median())

print("\n(Missing Values)")
print(df.isnull().sum())


(Missing Values)
ph                 0
Hardness           0
Solids             0
Chloramines        0
Sulfate            0
Conductivity       0
Organic_carbon     0
Trihalomethanes    0
Turbidity          0
Potability         0
dtype: int64


แยกข้อมูล Features และ Target

In [ ]:
X = df.drop('Potability', axis=1)
y = df['Potability']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Feature Scaling

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Train Model

In [ ]:
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
import joblib

In [ ]:
rf_model = RandomForestClassifier(random_state=42)
svm_model = SVC(random_state=42)
knn_model = KNeighborsClassifier()

In [ ]:
ensemble_model = VotingClassifier(
    estimators=[('rf', rf_model), ('svm', svm_model), ('knn', knn_model)],
    voting='hard'
)

In [ ]:
ensemble_model.fit(X_train_scaled, y_train)

VotingClassifier(estimators=[('rf', RandomForestClassifier(random_state=42)),
                             ('svm', SVC(random_state=42)),
                             ('knn', KNeighborsClassifier())])

In [ ]:
y_pred = ensemble_model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy : {accuracy * 100:.2f}%")

Accuracy : 68.75%


In [ ]:
joblib.dump(ensemble_model, 'water_ensemble_model.pkl')
joblib.dump(scaler, 'water_scaler.pkl')

['water_scaler.pkl']

Train Model NN

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt

In [ ]:
data_augmentation = tf.keras.Sequential([
  layers.RandomFlip("horizontal_and_vertical"),
  layers.RandomRotation(0.1),
  layers.RandomZoom(0.1),
])

In [ ]:
num_classes = len(class_names)

In [ ]:
model = models.Sequential([
  data_augmentation,
  layers.Rescaling(1./255, input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
  layers.Conv2D(16, 3, padding='same', activation='relu'),
  layers.MaxPooling2D(),
  layers.Conv2D(32, 3, padding='same', activation='relu'),
  layers.MaxPooling2D(),
  layers.Conv2D(64, 3, padding='same', activation='relu'),
  layers.MaxPooling2D(),
  layers.Flatten(),
  layers.Dense(128, activation='relu'),
  layers.Dense(num_classes, activation='softmax')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/preprocessing/data_layer.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [ ]:
model.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
              metrics=['accuracy'])

In [ ]:
epochs = 30
history = model.fit(
  train_dataset,
  validation_data=val_dataset,
  epochs=epochs
)

Epoch 1/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 6s 577ms/step - accuracy: 0.1875 - loss: 1.6256 - val_accuracy: 0.3500 - val_loss: 1.4483
Epoch 2/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 476ms/step - accuracy: 0.3625 - loss: 1.4867 - val_accuracy: 0.6000 - val_loss: 1.3282
Epoch 3/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 878ms/step - accuracy: 0.6750 - loss: 1.2701 - val_accuracy: 0.6500 - val_loss: 1.1523
Epoch 4/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 461ms/step - accuracy: 0.7500 - loss: 0.9854 - val_accuracy: 0.7500 - val_loss: 0.7790
Epoch 5/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 468ms/step - accuracy: 0.7375 - loss: 0.8959 - val_accuracy: 0.8000 - val_loss: 0.5714
Epoch 6/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 468ms/step - accuracy: 0.7625 - loss: 0.8185 - val_accuracy: 0.7500 - val_loss: 0.8156
Epoch 7/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 479ms/step - accuracy: 0.7875 - loss: 0.6402 - val_accuracy: 0.8500 - val_loss: 0.4953
Epoch 8/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 474ms/step - accuracy: 0.8375 - loss: 0.5106 - val_accuracy: 0.8000 - val_loss:

In [ ]:
model_save_path = 'football_league_model.h5'
model.save(model_save_path)